# Huấn luyện FER2013 trên Google Colab
### Dùng đúng pipeline SVM hiện tại, tải model về ứng dụng local

- **Created**: 2026-11-25T09:21:36+07:00
- **Last Updated**: 2026-11-25T12:39:18+07:00
- **Author**: Nhóm dự án FER
- **Objective**: Chạy script huấn luyện trên Colab và lưu model hoàn tất vào Google Drive.

---

## References & Consumed Artifacts

- **Source Code**: [Train CLI](../src/training/train.py), [đóng gói](../src/training/package_colab.py), [kiểm tra model](../src/emotion_recognition/svm_fer_classifier.py).
- **Artifacts**: [Thư mục model local](../models/), [ảnh train](../data/raw/train/). Model mới được ghi vào thư mục lần chạy riêng trên Drive.
- **Governing Rule**: [Huấn luyện bằng script](../agents/rules/LOGGING_CHECKPOINT_RULES.md).
- **References**: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [tương thích model scikit-learn](https://scikit-learn.org/stable/model_persistence.html).

SVM này chạy CPU; chọn GPU không tự tăng tốc. Colab không bảo đảm nhanh hơn máy local hoặc chạy liên tục vô hạn. Script không có resume trong lúc SVM đang fit. Drive chỉ giữ được model sau khi fit và ghi file hoàn tất.


## 1. Upload hoặc dùng lại gói đã giải nén

Dùng lại file `.tmp/fer2013_colab.zip` cũ. Nếu ZIP chưa nằm trong `/content`, ô dưới sẽ mở hộp chọn file. Nếu dự án đã giải nén đầy đủ từ lần chạy trước, ô này dùng lại ngay và không hỏi upload. Không cần giải nén bằng tay.


In [3]:
import json
import sys
import zipfile
from pathlib import Path
from google.colab import files

CONTENT_DIR = Path("/content")
PROJECT_DIR = CONTENT_DIR / "fer_lab"
manifest_path = PROJECT_DIR / "colab_manifest.json"
if manifest_path.is_file():
    print("Reusing the extracted project:", PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        raise RuntimeError("Project folder is incomplete. Use a fresh runtime; no files were deleted.")
    archive = CONTENT_DIR / "fer2013_colab.zip"
    if not archive.is_file():
        print("Select fer2013_colab.zip from your computer.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Select exactly one ZIP bundle.")
        archive = Path(next(iter(uploaded)))
        if not archive.is_absolute():
            archive = CONTENT_DIR / archive
        del uploaded
    if not archive.is_file():
        raise FileNotFoundError(f"Upload did not create the selected file: {archive}")
    with zipfile.ZipFile(archive) as bundle:
        required = {"colab_manifest.json", "requirements-colab.txt", "src/training/train.py"}
        if not required.issubset(bundle.namelist()):
            raise ValueError("This is not the FER project bundle. Select fer2013_colab.zip.")
        for member in bundle.infolist():
            (PROJECT_DIR / member.filename).resolve().relative_to(PROJECT_DIR.resolve())
        bundle.extractall(PROJECT_DIR)

manifest = json.loads(manifest_path.read_text())
if not (PROJECT_DIR / "src/training/train.py").is_file() or not (PROJECT_DIR / "data/raw/train").is_dir():
    raise RuntimeError("Extracted project is incomplete; use a fresh runtime.")
print("Colab kernel:", sys.version.split()[0])
print("Training Python:", ".".join(manifest["python_major_minor"]))
print("Training images:", manifest["training_images"])


Select fer2013_colab.zip from your computer.


KeyboardInterrupt: 

## 2. Tạo môi trường train khớp máy local

Kernel Colab có thể là Python 3.13. Ô này dùng uv tải Python 3.12 và tạo môi trường riêng theo phiên bản trong gói, rồi cài đúng các thư viện đã chốt. Kernel vẫn giữ nguyên; script train và kiểm tra model chạy bằng `TRAIN_PYTHON`. Cần kết nối mạng để tải Python/thư viện. Nếu có lỗi cài đặt, dừng tại đây, không chạy ô train.


In [ ]:
import subprocess

training_version = ".".join(manifest["python_major_minor"])
TRAIN_ENV = CONTENT_DIR / ("fer_train_py" + training_version.replace(".", ""))
TRAIN_PYTHON = TRAIN_ENV / "bin/python"
if not TRAIN_PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run([
        sys.executable, "-m", "uv", "venv", "--python", training_version,
        "--seed", str(TRAIN_ENV),
    ], check=True)

actual_version = subprocess.check_output([
    str(TRAIN_PYTHON), "-c",
    "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')",
], text=True).strip()
if actual_version != training_version:
    raise RuntimeError(f"Training environment is Python {actual_version}, expected {training_version}.")
subprocess.run([
    str(TRAIN_PYTHON), "-m", "pip", "install", "-r",
    str(PROJECT_DIR / "requirements-colab.txt"),
], check=True)
environment_json = subprocess.check_output([
    str(TRAIN_PYTHON), "-c",
    "import json, platform; from importlib.metadata import version; print(json.dumps({'python_version': platform.python_version(), 'dependencies': {name: version(name) for name in ['numpy', 'scikit-learn', 'scipy', 'joblib', 'Pillow', 'opencv-python-headless']}}))",
], text=True)
(PROJECT_DIR / "training_environment.json").write_text(environment_json)
print("Training interpreter:", TRAIN_PYTHON)
print(environment_json)


## 3. Kết nối Drive để giữ model hoàn tất

Chấp thuận kết nối Google Drive của bạn. Mỗi lần chạy tạo thư mục riêng dưới `MyDrive/FER_Lab`; không ghi đè lần chạy cũ. Nếu không gắn Drive được, dừng tại đây thay vì chỉ lưu model trong ổ tạm của Colab.


In [ ]:
import shutil
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/FER_Lab") / datetime.now().strftime("%Y%m%d_%H%M%S_fer")
RUN_DIR.mkdir(parents=True, exist_ok=False)
MODEL_PATH = RUN_DIR / "fer.joblib"
shutil.copy2(PROJECT_DIR / "colab_manifest.json", RUN_DIR / "colab_manifest.json")
shutil.copy2(PROJECT_DIR / "training_environment.json", RUN_DIR / "training_environment.json")
print("Model will be saved to:", MODEL_PATH)


## 4. Chạy script huấn luyện

Ô này gọi đúng CLI local, dùng ảnh mặt cắt sẵn. Chỉ train từ `train`, không dùng `test`. Log lưu cùng model trên Drive. Khi báo `Training SVM ...`, phép fit có thể im lặng khá lâu; không nhấn chạy lại ô khi tiến trình cũ còn chạy.


In [ ]:
if "TRAIN_ENV" not in globals():
    raise RuntimeError("Run step 2 (training environment setup) successfully before training.")
TRAIN_PYTHON = Path(TRAIN_ENV) / "bin/python"
if not TRAIN_PYTHON.is_file():
    raise RuntimeError(f"Training interpreter is missing: {TRAIN_PYTHON}. Rerun step 2 in this runtime before training.")
if not PROJECT_DIR.is_dir():
    raise RuntimeError("Project directory is missing. Rerun step 1 before training.")

import re
from tqdm.auto import tqdm


command = [
    str(TRAIN_PYTHON), "-u", "-m", "src.training.train",
    "--data-dir", str(PROJECT_DIR / "data/raw/train"),
    "--cropped-faces", "--output", str(MODEL_PATH),
]

with (RUN_DIR / "train.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    
    # Tạo thanh tiến trình
    pbar = tqdm(desc="Đang train", unit=" bước", dynamic_ncols=True)
    
    for line in process.stdout:
        log.write(line)
        log.flush()
        
        # Tự động bắt dạng "x/y" (vd: 500/28709) nếu log có in ra để kéo thanh % chính xác
        match = re.search(r"(\d+)\s*/\s*(\d+)", line)
        if match:
            cur, total = int(match.group(1)), int(match.group(2))
            if pbar.total != total:
                pbar.total = total
            pbar.n = cur
            pbar.refresh()
        else:
            pbar.update(1)
            # In log ra phía trên thanh tiến trình để không bị đè chữ
            tqdm.write(line.rstrip())
            
    pbar.close()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Training failed. Read {RUN_DIR / 'train.log'} before using any artifact.")
print("Training completed:", MODEL_PATH)

## 5. Kiểm tra model và tải về

Chỉ chạy ô này sau khi ô train hoàn tất thành công. Kiểm tra nạp model và nhãn trước khi tải. Model cũng nằm trên Drive để có thể tải lại sau khi Colab hết phiên.


In [ ]:
if return_code != 0 or not MODEL_PATH.is_file():
    raise RuntimeError("No successfully trained model to download.")
subprocess.run([
    str(TRAIN_PYTHON), "-c",
    "import sys; from pathlib import Path; from src.emotion_recognition.svm_fer_classifier import load_classifier; model = load_classifier(Path(sys.argv[1])); print('Verified labels:', model.classes_)",
    str(MODEL_PATH),
], cwd=PROJECT_DIR, check=True)
files.download(str(MODEL_PATH))


## 6. Dùng model trên máy

Đặt file vừa tải vào `models/fer_colab.joblib` trong dự án. Giữ model local cũ nếu còn cần so sánh. Mở PowerShell tại thư mục gốc:

```powershell
$env:FER_MODEL_PATH = (Resolve-Path models/fer_colab.joblib).Path
.venv\Scripts\python -m uvicorn src.api.main:app --port 8000
```

Nếu backend đang chạy, dừng bằng Ctrl+C trước khi chạy lệnh trên. Mở `http://127.0.0.1:8000/` để test ảnh mới. Notebook chỉ kiểm tra tính tương thích, chưa đo Accuracy/Macro-F1 hoặc thực hiện E1–E4.
